# 效果演示

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import griddata
from xyplot import XyPlot

# Define data reading function
def read_data(file):
    with open(file, 'r') as f:
        line_list = list(filter(lambda x: x[0] != '#', f.readlines()))
    variables = line_list[1].strip().split("=")[1].replace('"', '').split(",")
    data_list = []
    for line in line_list[6:]:
        try:
            line_data = [float(i) for i in line.strip().split(" ") if i]
        except:
            continue
        data_list.append(line_data)
    return pd.DataFrame(columns=variables, data=data_list)

# Prepare data
# Note: Assuming the data file exists, otherwise replace with actual file path
try:
    file = r"P-L1-IMM-SWMF_20221018004619_0005M_SWMF.dat"
    df = read_data(file)
    
    # Grid data preparation
    x, y = df['X [R]'].values, df['Y [R]'].values
    xx = yy = np.linspace(-6.5, 6.5, 1000)
    X, Y = np.meshgrid(xx, yy)
    grid_data = griddata((x, y), df[df.keys()[7]].values, (X, Y), method="linear")
    U = griddata((x, y), df[df.keys()[8]].values, (X, Y), method="linear")
    V = griddata((x, y), df[df.keys()[9]].values, (X, Y), method="linear")
    
    # Configure plotting parameters
    cfg = dict(
        title=dict(args="Flow Field Visualization", loc='left'),
        xlabel=dict(args="X [R]", c='k'),
        ylabel=dict(args="Y [R]", c='k'),
        streamplot=dict(args=(X, Y, U, V), density=1.5, linewidth=0.5, arrowsize=0.9, arrowstyle='->'),
        aspect=True,
        Branch=dict(
            contourf=dict(
                init=dict(args=(X, Y, grid_data), levels=np.linspace(0, 30, 50), extend="both", cmap=dict(
                                init=dict(
                                    name='chaos',
                                    colors=['black', 'purple', 'blue', 'cyan', 'green', 'yellow', 'orange', 'red'], N=100),
                                under='k', over='r'),
                        ),
                cbar=dict(
                    init=dict(shrink=0.8, ticks=np.linspace(0, 30, 11), orientation='horizontal'),
                    ax=dict(title=dict(args='Value Range', c='k'), xlabel='Value', ylabel='',)
                )
            ),
            patches=dict(
                wedge=(
                    dict(center=(0, 0), r=1, theta1=90, theta2=270, color='k',),
                    dict(center=(0, 0), r=1, theta1=-90, theta2=90, edgecolor='k', facecolor='w'),
                )
            ),
        )
    )
    
    # Simple sine function example
    x_sin = np.linspace(-np.pi, np.pi, 100)
    y_sin = np.sin(x_sin)
    cfg_sin = dict(
        title=dict(args="Sine Function", loc='center'),
        xlabel=dict(args="x", c='k'),
        ylabel=dict(args=r'$\sin(x)$', c='k'),
        plot=dict(args=(x_sin, y_sin), label=r'$y = \sin(x)$', c='r', lw=2),
        grid=dict(linestyle=':', color='gray'),
        legend=dict(loc='upper right'),
    )
    
    # Set figure parameters
    fig_dict = dict(
        height=10, width=15,
        title=dict(args='XyPlot Demo'),
    )
    
    # Create multi-subplot layout
    axes_dict = dict(
        set_fig=fig_dict,
        axes=dict(
            init=(dict(args=(1, 2, 1), ), 122),
            axes=(cfg, cfg_sin)
        )
    )
    
    # Create and display the chart
    xy_plot = XyPlot(**axes_dict)
    xy_plot.show()
    
except Exception as e:
    # If data file doesn't exist, only show simple sine plot
    print(f"Note: {e}")
    print("Showing basic sine plot example...")
    
    x = np.linspace(-np.pi, np.pi, 100)
    y = np.sin(x)
    
    set_fig_dict = dict(height=8, width=10)
    axes_dict = dict(
        plot=dict(args=(x, y), label='y=sin(x)', c='r', lw=2),
        title=r'y=sin(x)',
        grid=dict(linestyle=':', color='gray'),
        xlabel="x",
        ylabel="y",
        legend=dict(loc='upper right'),
    )
    
    cfg = dict(set_fig=set_fig_dict, axes=axes_dict)
    xyplt = XyPlot(**cfg)
    xyplt.show()
